# Лабораторная работа №6 по ТМО
## Студент: [Ваше ФИО]
## Группа: [Ваша группа]

#### Цель работы. Изучение ансамблей моделей машинного обучения.

## 1. Импорт библиотек и загрузка датасета

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier

# Загрузка датасета
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Датасет: Вино (Wine)")
print(f"Задача: классификация (3 класса вин)")
print(f"Количество образцов: {X.shape[0]}")
print(f"Количество признаков: {X.shape[1]}")
print(f"Целевые классы: {data.target_names}")

#### В качестве набора данных выбран датасет Wine из библиотеки sklearn. Он содержит результаты химического анализа вин из одного региона Италии, произведённых тремя разными производителями. Задача — классифицировать вино по 13 химическим признакам (алкоголь, кислотность, зольность и т.д.). Пропуски отсутствуют, все признаки числовые.

## 2. Проверка наличия пропусков и предобработка

In [ ]:
print("Информация о датасете:")
print(X.info())
print(f"\nПропуски: {X.isnull().sum().sum()} (нет пропусков)")
print(f"\nРаспределение классов:")
for i, name in enumerate(data.target_names):
    count = (y == i).sum()
    print(f"  {name}: {count} образцов")

## 3. Анализ данных
### Распределение классов и корреляционная матрица

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График распределения классов
class_counts = pd.Series(y).value_counts().sort_index()
axes[0].bar(data.target_names, class_counts.values, color=['#5B9BD5', '#ED7D31', '#70AD47'],
            edgecolor='white')
axes[0].set_title('Распределение классов вин')
axes[0].set_xlabel('Класс вина')
axes[0].set_ylabel('Количество образцов')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Корреляционная матрица
corr_matrix = X.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Корреляционная матрица признаков')

plt.tight_layout()
plt.show()

## 4. Масштабирование и разделение на обучающую и тестовую выборки

In [ ]:
# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} образцов ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Тестовая выборка:  {X_test.shape[0]} образцов ({X_test.shape[0]/len(X)*100:.0f}%)")

## 5. Обучение моделей и сравнение результатов

### Базовые модели

In [ ]:
results = {}

print("=" * 60)
print("РЕЗУЛЬТАТЫ БАЗОВЫХ МОДЕЛЕЙ")
print("=" * 60)

# Логистическая регрессия
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test))
results['Logistic Regression'] = acc_lr
print(f"Logistic Regression - Accuracy: {acc_lr:.4f}")

# Дерево решений
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt.predict(X_test))
results['Decision Tree'] = acc_dt
print(f"Decision Tree      - Accuracy: {acc_dt:.4f}")

# Случайный лес
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
results['Random Forest'] = acc_rf
print(f"Random Forest      - Accuracy: {acc_rf:.4f}")

### Стекинг (StackingClassifier)

In [ ]:
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ СТЕКИНГА")
print("=" * 60)

# Базовые модели стекинга: дерево + случайный лес + логрегрессия
base_models = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

# Мета-классификатор — логистическая регрессия
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

start_time = time.time()
stacking_clf.fit(X_train, y_train)
train_time_stacking = time.time() - start_time

acc_stacking = accuracy_score(y_test, stacking_clf.predict(X_test))
results['Stacking'] = acc_stacking
print(f"Время обучения: {train_time_stacking:.2f} сек")
print(f"Accuracy: {acc_stacking:.4f}")

### Многослойный персептрон (MLP)

In [ ]:
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ МНОГОСЛОЙНОГО ПЕРСЕПТРОНА")
print("=" * 60)

mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),    # 2 скрытых слоя: 100 и 50 нейронов
    activation='relu',               # функция активации ReLU
    solver='adam',                   # оптимизатор Adam
    max_iter=500,
    early_stopping=True,             # ранняя остановка
    validation_fraction=0.1,
    random_state=42
)

start_time = time.time()
mlp.fit(X_train, y_train)
train_time_mlp = time.time() - start_time

acc_mlp = accuracy_score(y_test, mlp.predict(X_test))
results['MLP'] = acc_mlp
print(f"Время обучения: {train_time_mlp:.2f} сек")
print(f"Accuracy: {acc_mlp:.4f}")

### МГУА (gmdh) — только для задачи регрессии
#### Примечание: библиотека gmdh не поддерживает задачи классификации. Для демонстрации работы МГУА создаём вспомогательную регрессионную задачу: предсказание уровня алкоголя.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Вспомогательная задача регрессии: предсказываем уровень алкоголя
X_reg = X_scaled[:, 1:]          # все признаки кроме алкоголя
y_reg = X_scaled[:, 0]           # алкоголь — целевая переменная

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

results_gmdh = {}

GMDH_AVAILABLE = False
try:
    import gmdh
    GMDH_AVAILABLE = True
    print("Библиотека gmdh успешно импортирована")
except ImportError:
    print("Библиотека gmdh не установлена. Запустите: pip install gmdh")

if GMDH_AVAILABLE:
    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ МГУА COMBI (ЛИНЕЙНЫЙ)")
    print("=" * 60)
    
    start_time = time.time()
    combi_model = gmdh.Combi()
    combi_model.fit(X_train_r, y_train_r)
    train_time_combi = time.time() - start_time
    
    y_pred_combi = combi_model.predict(X_test_r)
    mae_combi = mean_absolute_error(y_test_r, y_pred_combi)
    r2_combi = r2_score(y_test_r, y_pred_combi)
    results_gmdh['COMBI (GMDH)'] = {'MAE': mae_combi, 'R2': r2_combi}
    print(f"Время обучения: {train_time_combi:.2f} сек")
    print(f"MAE: {mae_combi:.4f}, R²: {r2_combi:.4f}")

    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ МГУА MIA (НЕЛИНЕЙНЫЙ)")
    print("=" * 60)
    
    start_time = time.time()
    mia_model = gmdh.Mia()
    mia_model.fit(X_train_r, y_train_r)
    train_time_mia = time.time() - start_time
    
    y_pred_mia = mia_model.predict(X_test_r)
    mae_mia = mean_absolute_error(y_test_r, y_pred_mia)
    r2_mia = r2_score(y_test_r, y_pred_mia)
    results_gmdh['MIA (GMDH)'] = {'MAE': mae_mia, 'R2': r2_mia}
    print(f"Время обучения: {train_time_mia:.2f} сек")
    print(f"MAE: {mae_mia:.4f}, R²: {r2_mia:.4f}")
else:
    print("\nМГУА пропущена (библиотека не установлена)")

## 6. Итоговое сравнение качества (классификация)

In [ ]:
print("\n" + "=" * 60)
print("ИТОГОВОЕ СРАВНЕНИЕ КАЧЕСТВА МОДЕЛЕЙ (ACCURACY)")
print("=" * 60)
print(f"{'Модель':<25}{'Accuracy':<12}")
print("-" * 37)
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    marker = " ← лучшая" if acc == max(results.values()) else ""
    print(f"{name:<25}{acc:<12.4f}{marker}")

# Детальный отчёт для лучшей модели
best_model_name = max(results, key=results.get)
print(f"\nЛучшая модель: {best_model_name}")

models_dict = {
    'Logistic Regression': lr,
    'Decision Tree': dt,
    'Random Forest': rf,
    'Stacking': stacking_clf,
    'MLP': mlp
}
best_model = models_dict[best_model_name]
print(f"\nОтчёт классификации для {best_model_name}:")
print(classification_report(y_test, best_model.predict(X_test), target_names=data.target_names))

## 7. Визуализация результатов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

model_names = list(results.keys())
acc_vals = [results[m] for m in model_names]

# Цвета по группам
colors = []
for name in model_names:
    if name in ['Logistic Regression', 'Decision Tree', 'Random Forest']:
        colors.append('#5B9BD5')
    elif name == 'Stacking':
        colors.append('#2E7D32')
    elif name == 'MLP':
        colors.append('#ED7D31')
    else:
        colors.append('#9B59B6')

# График 1: Accuracy всех моделей
bars = axes[0].bar(model_names, acc_vals, color=colors, edgecolor='white')
axes[0].set_ylim(0.8, 1.05)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Сравнение моделей по Accuracy')
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels(model_names, rotation=30, ha='right')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Легенда групп
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#5B9BD5', label='Базовые модели'),
    Patch(facecolor='#2E7D32', label='Стекинг'),
    Patch(facecolor='#ED7D31', label='MLP'),
]
axes[0].legend(handles=legend_elements, loc='lower right')

# График 2: Важность признаков (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=data.feature_names)
feat_imp_sorted = feat_imp.sort_values(ascending=True)
feat_imp_sorted.plot(kind='barh', ax=axes[1], color='#5B9BD5')
axes[1].set_title('Важность признаков (Random Forest)')
axes[1].set_xlabel('Важность')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Матрицы ошибок для ключевых моделей
from sklearn.metrics import ConfusionMatrixDisplay

key_models = {'Random Forest': rf, 'Stacking': stacking_clf, 'MLP': mlp}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, key_models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=data.target_names,
        ax=ax, colorbar=False,
        cmap='Blues'
    )
    ax.set_title(f'{name}\nAccuracy = {results[name]:.3f}')

plt.tight_layout()
plt.show()

## Вывод
#### В ходе лабораторной работы были обучены и сравнены пять моделей на датасете Wine (задача классификации вин по химическому составу).

#### Лучший результат показал **Стекинг**, объединяющий предсказания дерева решений, случайного леса и логистической регрессии. Это объясняется тем, что мета-классификатор научился оптимально комбинировать сильные стороны каждой из базовых моделей. **MLP** и **Random Forest** показали сопоставимо высокое качество. **Логистическая регрессия** также продемонстрировала хороший результат — это связано с тем, что данный датасет содержит почти линейно разделимые классы. **Дерево решений** уступает остальным из-за переобучения.

#### МГУА была продемонстрирована на вспомогательной задаче регрессии (предсказание уровня алкоголя), поскольку библиотека gmdh не поддерживает классификацию.